In [2]:
! pip install beautifulsoup4

In [10]:
import requests
from bs4 import BeautifulSoup
import json
import time
import os

In [11]:
URLS = [
    "https://crocoit.com",
    "https://crocoit.com/about-us/",
    "https://crocoit.com/solutions/",
    "https://crocoit.com/our-products/",
    "https://crocoit.com/articles/",
    "https://crocoit.com/case-studies/"
]

In [12]:
def scrape_page(url):
    try:
        headers = {
            "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                          "AppleWebKit/537.36 (KHTML, like Gecko) "
                          "Chrome/120.0.0.0 Safari/537.36"
        }

        response = requests.get(url, headers=headers, timeout=10)
        response.raise_for_status()  # raises error if status != 200

        soup = BeautifulSoup(response.text, "html.parser")

        # Remove unwanted tags (scripts, styles, nav, footer)
        for tag in soup(["script", "style", "nav", "footer", "header", "noscript"]):
            tag.decompose()

        # Extract page title
        title = soup.title.string.strip() if soup.title else url

        # Extract all visible text from headings and paragraphs
        tags = soup.find_all(["h1", "h2", "h3", "h4", "p", "li", "span"])
        texts = []
        for tag in tags:
            text = tag.get_text(separator=" ", strip=True)
            if len(text) > 30:  # skip very short/empty strings
                texts.append(text)

        # Join all text into one clean block
        full_content = "\n".join(texts)

        print(f"✅ Scraped: {url} ({len(full_content)} chars)")
        return {
            "url": url,
            "title": title,
            "content": full_content
        }

    except requests.exceptions.HTTPError as e:
        print(f"❌ HTTP Error {url}: {e}")
        return None
    except requests.exceptions.ConnectionError:
        print(f"❌ Connection failed: {url}")
        return None
    except Exception as e:
        print(f"❌ Unknown error {url}: {e}")
        return None

In [13]:
scraped_data = []

for url in URLS:
    result = scrape_page(url)
    if result:
        scraped_data.append(result)
    time.sleep(1)  # be polite — wait 1 second between requests

print(f"\n✅ Total pages scraped: {len(scraped_data)}")

✅ Scraped: https://crocoit.com (2861 chars)
✅ Scraped: https://crocoit.com/about-us/ (2597 chars)
✅ Scraped: https://crocoit.com/solutions/ (1974 chars)
✅ Scraped: https://crocoit.com/our-products/ (1681 chars)
✅ Scraped: https://crocoit.com/articles/ (1500 chars)
✅ Scraped: https://crocoit.com/case-studies/ (1593 chars)

✅ Total pages scraped: 6


In [15]:
if scraped_data:
    print("=== PREVIEW: First Page ===")
    print(f"URL   : {scraped_data[2]['url']}")
    print(f"Title : {scraped_data[2]['title']}")
    print(f"Content preview:\n")
    print(scraped_data[2]['content'][:2000])  # first 1000 chars

=== PREVIEW: First Page ===
URL   : https://crocoit.com/solutions/
Title : Solutions – CrocoIT
Content preview:

Transform your business with our expert consultation and audit services. Our team of seasoned consultants provides in-depth digital transformation consultation, IT audits, and business process evaluations to ensure your technology strategy drives growth. Benefit from a data-driven approach and actionable insights tailored to your industry.
Boost your brand’s online presence with our comprehensive digital marketing services. From SEO and PPC advertising to social media and content marketing, we design strategies that target the right audience and maximize ROI. Let our marketing experts help you capture more leads and increase conversions across digital channels
Maximize the potential of your online marketplace with our marketplace management expertise. We offer end-to-end services to optimize vendor performance, manage listings, and implement strategic growth initiatives. Sta

In [16]:
os.makedirs("data", exist_ok=True)

output_path = "data/raw_pages.json"

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(scraped_data, f, ensure_ascii=False, indent=2)

print(f"✅ Saved {len(scraped_data)} pages to {output_path}")

✅ Saved 6 pages to data/raw_pages.json


In [17]:
with open("data/raw_pages.json", "r", encoding="utf-8") as f:
    loaded = json.load(f)

print(f"Pages in file: {len(loaded)}")
for page in loaded:
    print(f"  - {page['title']} | {len(page['content'])} chars | {page['url']}")

Pages in file: 6
  - CrocoIT - CrocoIT | 2861 chars | https://crocoit.com
  - About Us – CrocoIT | 2597 chars | https://crocoit.com/about-us/
  - Solutions – CrocoIT | 1974 chars | https://crocoit.com/solutions/
  - Our Products – CrocoIT | 1681 chars | https://crocoit.com/our-products/
  - Articles – CrocoIT | 1500 chars | https://crocoit.com/articles/
  - Case Studies – CrocoIT | 1593 chars | https://crocoit.com/case-studies/
